# 02 — Data Quality, Schema Validation & Missingness Audit

**Project:** Justice for Rural Communities Observatory (JRC Observatory)  
**Purpose:** Validate all 12 JSON Schemas, audit `P5` synthetic test datasets, and inventory/validate all empirical `P1` files in `data/approved-public/`.

In [ ]:
import json
import sys
from pathlib import Path
from jsonschema import Draft202012Validator, FormatChecker

ROOT = Path('..').resolve() if Path('../schemas').exists() else Path('.').resolve()
sys.path.insert(0, str(ROOT / 'src' / 'validation'))
import validate_all

schemas = validate_all.validate_schemas()
synth_count = validate_all.validate_synthetic_data(schemas)
source_count = validate_all.validate_source_register()
print(f'Schemas valid: {len(schemas)} | P5 synthetic records valid: {synth_count} | Sources valid: {source_count}')

In [ ]:
approved_files = sorted((ROOT / 'data' / 'approved-public').glob('*.json'))
print(f'Empirical P1 files in data/approved-public/: {len(approved_files)}')

schema_map = {
    'DS02': 'justice-service-locations.schema.json',
    'DS05': 'legal-aid.schema.json',
}

total_schema_records = 0
for p in approved_files:
    data = json.loads(p.read_text(encoding='utf-8'))
    tier = data.get('publication_tier')
    ds_id = data.get('dataset_id', 'AGGREGATE_TABLE')
    recs = data.get('records')
    if isinstance(recs, list):
        if ds_id in schema_map:
            v = Draft202012Validator(schemas[schema_map[ds_id]], format_checker=FormatChecker())
            for r in recs:
                v.validate(r)
        pending = sum(1 for r in recs if r.get('verification_status') == 'pending_verification')
        total_schema_records += len(recs)
        print(f'  - {p.name}: tier={tier}, dataset={ds_id}, records={len(recs)}, pending_verification={pending}')
    else:
        keys = [k for k in ('tables', 'provinces', 'pilot_rows', 'national_summary') if k in data]
        print(f'  - {p.name}: tier={tier}, period={data.get("observation_period")}, sections={keys}')
print(f'Total schema-validated empirical P1 records: {total_schema_records}')